# 02 · Eventos sintéticos — camada Silver

*Personas, cliques e o problema de não ter dados de usuário*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## O elefante na sala: não temos cliques reais

Um recomendador personalizado aprende com **comportamento** (quem leu o quê). Esse
dado é privado — fica no *analytics* do portal. Então **simulamos**, de forma explícita
e reproduzível (`seed`):

- 8 **personas** (politizado, investidor, torcedor, carioca, cultural, ...), cada uma
  com preferência por editorias — e cada usuário é uma variação da sua persona;
- **subeditorias favoritas** (ex.: `esportes/futebol`) → o "gosto fino" que só a
  filtragem colaborativa consegue capturar;
- **viés de popularidade** (matérias que "viralizam") e **de recência** (notícia velha
  perde interesse — meia-vida de 12 h).

> Em produção, esta tabela viria de eventos reais (clickstream). **O resto do pipeline
> não muda** — é por isso que a simulação é só uma etapa isolada.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

### `simulation.py`

In [ ]:
# src/news_recsys/simulation.py  (somente leitura — o código de verdade está em src/)
"""Usuários e cliques SINTÉTICOS sobre matérias reais.

Não temos clickstream do O Globo — então simulamos. Cada usuário tem uma *persona* (preferência
por editorias), um gosto fino por subeditorias e sofre viés de popularidade e de recência. O
resultado imita a estrutura de um log real de eventos (user, matéria, timestamp), e é com ele
que os modelos são treinados. Em produção esse dado viria do analytics (eventos de clique).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

PERSONAS: dict[str, dict[str, float]] = {
    "politizado": {"politica": 0.55, "brasil": 0.15, "blogs": 0.15, "opiniao": 0.1, "mundo": 0.05},
    "investidor": {"economia": 0.6, "politica": 0.15, "mundo": 0.15, "blogs": 0.1},
    "torcedor": {"esportes": 0.7, "rio": 0.15, "cultura": 0.15},
    "carioca": {"rio": 0.45, "cultura": 0.2, "esportes": 0.15, "brasil": 0.2},
    "cultural": {"cultura": 0.45, "ela": 0.3, "saude": 0.15, "outros": 0.1},
    "globalista": {"mundo": 0.55, "economia": 0.2, "politica": 0.15, "ciencia": 0.1},
    "bem-estar": {"saude": 0.45, "ela": 0.3, "ciencia": 0.15, "cultura": 0.1},
    "generalista": {},  # sem preferência: segue a distribuição do catálogo
}


@dataclass(frozen=True)
class SimulationConfig:
    n_users: int = 300
    mean_clicks_per_user: float = 18.0
    min_clicks_per_user: int = 4
    section_weight: float = 3.0  # força da preferência por editoria
    subsection_weight: float = 2.0  # gosto fino (subeditorias favoritas)
    popularity_weight: float = 1.0
    recency_half_life_h: float = 12.0
    noise: float = 0.15  # cliques "aleatórios"
    seed: int = 42


def _parse_ts(values: pd.Series) -> pd.Series:
    return pd.to_datetime(values, utc=True, format="mixed", errors="coerce")


def make_users(articles: pd.DataFrame, cfg: SimulationConfig) -> pd.DataFrame:
    """Um usuário por linha: persona + pesos por editoria + 2 subeditorias favoritas."""
    rng = np.random.default_rng(cfg.seed)
    sections = sorted(articles["section"].unique())
    base_mix = articles["section"].value_counts(normalize=True).reindex(sections).fillna(0.0)
    names = list(PERSONAS)
    rows = []
    for i in range(cfg.n_users):
        persona = names[i % len(names)] if i < len(names) else str(rng.choice(names))
        prefs = PERSONAS[persona] or base_mix.to_dict()
        weights = np.array([prefs.get(s, 0.0) for s in sections]) + 0.01
        weights = rng.dirichlet(weights / weights.sum() * 20)  # cada usuário é um pouco diferente
        subs = articles.loc[articles["section"].isin(_top(sections, weights, 2)), "subsection"]
        favorites = list(subs.value_counts().index[:6])
        fav = (
            list(rng.choice(favorites, size=min(2, len(favorites)), replace=False))
            if favorites
            else []
        )
        rows.append(
            {
                "user_id": f"u_{i:04d}",
                "persona": persona,
                "section_prefs": dict(zip(sections, np.round(weights, 4).tolist(), strict=True)),
                "favorite_subsections": fav,
            }
        )
    return pd.DataFrame(rows)


def _top(items: list[str], weights: np.ndarray, n: int) -> list[str]:
    return [items[i] for i in np.argsort(weights)[::-1][:n]]


def simulate_clicks(
    articles: pd.DataFrame, users: pd.DataFrame, cfg: SimulationConfig
) -> pd.DataFrame:
    """Gera o log de cliques: user_id, article_id, event_ts (+ persona para análise)."""
    rng = np.random.default_rng(cfg.seed + 1)
    arts = articles.reset_index(drop=True).copy()
    published = _parse_ts(arts["published_at"])
    t0, t1 = published.min(), published.max()
    span_h = max((t1 - t0).total_seconds() / 3600, 1.0)
    pub_h = ((published - t0).dt.total_seconds() / 3600).to_numpy()

    popularity = rng.lognormal(mean=0.0, sigma=1.0, size=len(arts))  # "matéria que viralizou"
    section_idx = arts["section"].to_numpy()
    subsection_idx = arts["subsection"].to_numpy()

    events = []
    for user in users.itertuples(index=False):
        n_clicks = max(cfg.min_clicks_per_user, int(rng.poisson(cfg.mean_clicks_per_user)))
        sec_w = np.array([user.section_prefs.get(s, 0.0) for s in section_idx])
        sub_bonus = np.isin(subsection_idx, user.favorite_subsections).astype(float)
        # momentos de leitura espalhados pela janela do catálogo
        read_h = np.sort(rng.uniform(0, span_h + 6, size=n_clicks))
        seen: set[int] = set()
        for t in read_h:
            age = t - pub_h
            available = age >= 0
            if not available.any():
                continue
            recency = np.where(
                available, 0.5 ** (np.clip(age, 0, None) / cfg.recency_half_life_h), 0
            )
            score = (
                (sec_w**cfg.section_weight + 1e-6)
                * (1 + cfg.subsection_weight * sub_bonus)
                * popularity**cfg.popularity_weight
                * recency
            )
            score[list(seen)] = 0.0
            if score.sum() <= 0:
                continue
            probs = (1 - cfg.noise) * score / score.sum()
            probs += cfg.noise * available / available.sum()
            probs[list(seen)] = 0.0
            probs /= probs.sum()
            idx = int(rng.choice(len(arts), p=probs))
            seen.add(idx)
            events.append(
                {
                    "user_id": user.user_id,
                    "article_id": arts.at[idx, "article_id"],
                    "event_ts": t0 + pd.Timedelta(hours=float(t)),
                    "persona": user.persona,
                }
            )
    out = pd.DataFrame(events, columns=["user_id", "article_id", "event_ts", "persona"])
    return out.sort_values(["user_id", "event_ts"]).reset_index(drop=True)


def simulate(
    articles: pd.DataFrame, cfg: SimulationConfig | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Atalho: (usuários, cliques)."""
    cfg = cfg or SimulationConfig()
    users = make_users(articles, cfg)
    return users, simulate_clicks(articles, users, cfg)

### Execução: janela de catálogo + simulação

Notícia envelhece rápido: usamos só as matérias dos **últimos 3 dias** (a "janela"
recomendável). A janela vira uma tabela para que treino e inferência usem o mesmo catálogo.

In [ ]:
import pandas as pd

from news_recsys.simulation import SimulationConfig, simulate

bronze = spark.table(settings.table("bronze_articles")).toPandas()
published = pd.to_datetime(bronze["published_at"], utc=True, format="mixed", errors="coerce")
articles = bronze[published >= published.max() - pd.Timedelta(days=3)].reset_index(drop=True)
print(f"janela: {len(articles)} de {len(bronze)} matérias (últimos 3 dias)")

cfg = SimulationConfig(n_users=settings.n_users, mean_clicks_per_user=settings.mean_clicks_per_user, seed=settings.seed)
users, clicks = simulate(articles, cfg)
print(f"{len(users)} usuários · {len(clicks)} cliques · {clicks.groupby('user_id').size().mean():.1f} cliques/usuário")
users.head(8)

### Execução: gravando as tabelas silver

In [ ]:
import json

from pyspark.sql import functions as F

spark.createDataFrame(articles[["article_id"]]).write.mode("overwrite").saveAsTable(settings.table("silver_catalog_window"))
users_out = users.assign(  # dict/list -> JSON/str: tipos simples e portáveis no Delta
    section_prefs=users["section_prefs"].map(json.dumps),
    favorite_subsections=users["favorite_subsections"].map(lambda v: ",".join(v)),
)
spark.createDataFrame(users_out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(settings.table("silver_users"))
(
    spark.createDataFrame(clicks)
    .withColumn("event_ts", F.col("event_ts").cast("timestamp"))
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(settings.table("silver_interactions"))
)
for t in ("silver_catalog_window", "silver_users", "silver_interactions"):
    print(f"{settings.table(t):55s} {spark.table(settings.table(t)).count():6d} linhas")

### Visualizando: cada persona lê coisas diferentes?

Se a simulação está boa, cada linha do mapa de calor tem "cor" concentrada nas
editorias da persona — é esse padrão que os modelos vão tentar aprender.

In [ ]:
import matplotlib.pyplot as plt

section = articles.set_index("article_id")["section"]
mix = pd.crosstab(clicks["persona"], clicks["article_id"].map(section), normalize="index")
mix = mix.loc[:, mix.max().sort_values(ascending=False).index[:10]]

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(mix.to_numpy(), cmap="Blues", aspect="auto", vmin=0, vmax=0.8)
ax.set_xticks(range(len(mix.columns)), mix.columns, rotation=40, ha="right")
ax.set_yticks(range(len(mix.index)), mix.index)
for (i, j), v in pd.DataFrame(mix.to_numpy()).stack().items():
    ax.text(j, i, f"{v:.0%}" if v >= 0.05 else "", ha="center", va="center", fontsize=8)
ax.set_title("Fração das leituras por editoria, por persona")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
display(fig)

### O que percebemos?

- O dado de comportamento é o **ativo mais valioso** de um recomendador — e o mais
  sensível (LGPD). Aqui ele é sintético e público.
- A simulação é **determinística** (`seed`): mesma entrada → mesmos cliques → métricas
  comparáveis entre execuções.
- O formato (user, item, timestamp) é o mesmo de um log real.

**Próximo:** `03_treino_mlflow`.